# 02 · Dask: comparación de configuraciones de workers y memoria
Se ejecuta el mismo trabajo (capa SILVER + GOLD) con distintos perfiles definidos en `config.yaml`:

| Perfil | Workers | Hilos/worker | Memoria/worker | Idea |
|---|---|---|---|---|
| A | 2 | 2 | 3 GiB | pocos workers con mucha memoria |
| B | 4 | 1 | 1.5 GiB | más procesos en paralelo, menos memoria cada uno |
| C | 1 | 4 | 6 GiB | referencia: un solo proceso con hilos (limitado por el GIL) |

Cada corrida genera `reports/dask-report-<perfil>.html` (reporte de rendimiento de Dask: evidencia para el documento).

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
from src.utils import load_config

cfg = load_config()
FIG = cfg["paths"]["reports_dir"] / "figures"; FIG.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.float_format", "{:,.3f}".format)

def guardar(nombre):
    plt.tight_layout(); plt.savefig(FIG / f"{nombre}.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
from src.dask_benchmark import main
res = main(["A", "B", "C"])     # puede tardar varios minutos con el dataset completo
res

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
res.set_index("profile")[["seconds_silver", "seconds_gold"]].plot.bar(stacked=True, ax=ax[0])
ax[0].set_title("Tiempo por etapa (s)"); ax[0].set_ylabel("segundos")
res.set_index("profile")["rows_per_second"].plot.bar(ax=ax[1], color="#55A868")
ax[1].set_title("Reseñas procesadas por segundo")
guardar("10_benchmark_dask")

### Interpretación (completar)
* El cálculo de sentimiento (VADER) es Python puro y está limitado por el GIL: **más procesos** (perfil B) suelen rendir más que **más hilos** (perfil C).
* Con poca memoria por worker (perfil B) pueden aparecer avisos de *spilling* a disco en el *merge* o en `drop_duplicates`: revisar el reporte HTML.
* Recomendación para el dataset completo: perfil ______ porque ______.